<center><h1 style="color:#173F8A;"> Metodos para Ecuaciones Diferenciales, IMT3410, 2024-2 </h1></center>
<h3 style="color:#173F8A;text-align:right;"> Instituto de Ingenieria Matematica y Computacional<br>  Pontificia Universidad Catolica de Chile <br>  </h3>
<h3 style="color:#0176DE;text-align:right;"> Profesor. Manuel A. Sanchez<br> </h3>
<hr style="border:2px solid #03122E"> </hr>

<!-- Palette colors UC: celeste:#0176DE, azul #173F8A, azul oscuro: #03122E, amarillo: #FEC60D, amarillo oscuro: #E3AE00 -->
<!--
<figure>
<img align ="right" src="IMClogo.png" alt="logo" width="250" height="400"><br><br><br><br><br>
</figure>
 -->
 
 <h2 style="color:#03122E;text-align:center;"> Capitulo 2. Metodos para Ecuaciones Diferenciales Parciales Elipticas<br> </h2>
<h3 style="color:#03122E;text-align:center;">             Metodos de Elementos Finitos <br> </h3>
<h3 style="color:#03122E;text-align:center;">             Problemas de adveccion-difusion <br> </h3>
<hr style="border:2px solid #03122E"> </hr>

In [1]:
# NGSolve Libraries
from netgen.geom2d import unit_square
from ngsolve import *
from ngsolve.webgui import Draw # para jupyter
#import netgen.gui
from netgen.geom2d import SplineGeometry
from pandas import DataFrame

## Adveccion-difusion
Find $u:\mathbb R \mapsto \mathbb R$ such that
\begin{align*}
-\nu \Delta u  + \nabla u \cdot \beta & = f, \quad \mbox{in } \Omega, \\
u & = g, \quad \mbox{on } \partial \Omega.
\end{align*}

**Primal variational form.** Find $u\in H_0^{1}(\Omega)$ such that:
\begin{align*}
\int_{\Omega} \left( \nu\nabla u \cdot \nabla v + (\beta\cdot \nabla u) v \right) dx = \int_{\Omega} f v dx 
\end{align*}

Consider $\Omega = (0,1)^{2}$ and boundary conditions:
$$
u = 1, \;\text{on } x=1,\quad u = 0, \;\text{on } x = 0,\quad \frac{\partial}{\partial y}u = \nabla u\cdot n = 0, \;\text{on } y = 0,y = 1.
$$

In [2]:
# parametros
nu = CF(0.000000001)
beta = (10.,0.)
f = 0
uexact = (exp(x/nu)-1.0)/(exp(1.0/nu)-1.0) 

h_max = 0.25

In [3]:
def solve_advdiff(order=1, h_max=0.1):
    ngmesh = unit_square.GenerateMesh(maxh=h_max)
    mesh = Mesh(ngmesh)
    fes = H1(mesh, order=1, dirichlet='right|left')
    u, v = fes.TnT()

    a = BilinearForm(fes)
    a += nu * grad(u)*grad(v) *dx
    a += grad(u)*beta * v *dx
    a.Assemble()

    L = LinearForm(fes)
    L += 0*v*dx
    L.Assemble()

    gfu = GridFunction(fes)
    gfu.Set(1.0, definedon=mesh.Boundaries("right"))
    gfu.vec.data += a.mat.Inverse(freedofs=fes.FreeDofs()) * L.vec
    return gfu

In [4]:
u1 = solve_advdiff(order=1, h_max=0.2)
u2 = solve_advdiff(order=1, h_max=0.05)
u3 = solve_advdiff(order=1, h_max=0.025)
u4 = solve_advdiff(order=2, h_max=0.025)

In [5]:
Draw(u1);
Draw(u2);
Draw(u3);
Draw(u4)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene

In [6]:
ngmesh = unit_square.GenerateMesh(maxh=0.05)
mesh = Mesh(ngmesh)
V = H1(mesh, order=1, dirichlet='bottom|right|top|left')
u, v = V.TnT()

nu = 0.01 # nu = 0.00001 y h = 0.1
beta = (1.,1.)
a = BilinearForm(V)
a += nu * grad(u)*grad(v) *dx
a += grad(u)*beta * v *dx

a.Assemble()

f = 1.
L = LinearForm(V)
L += f*v*dx
L.Assemble()

gfu = GridFunction(V)
gfu.vec.data = a.mat.Inverse(freedofs=V.FreeDofs()) * L.vec
Draw(gfu)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene

In [116]:
# NGSolve Libraries
from netgen.geom2d import unit_square
from ngsolve import *
from ngsolve.webgui import Draw # para jupyter
#import netgen.gui
from netgen.geom2d import SplineGeometry
from pandas import DataFrame

def solve_advdiff_2(order=1, h_max=0.1, uD = 1.0, nu=1, beta = (1,1) ):
    ngmesh = unit_square.GenerateMesh(maxh=h_max)
    mesh = Mesh(ngmesh)
    fes = H1(mesh, order=1, dirichlet='left')
    u, v = fes.TnT()

    a = BilinearForm(fes)
    a += nu * grad(u)*grad(v) *dx
    a += grad(u)*beta * v *dx
    a.Assemble()

    L = LinearForm(fes)
    L += 1*v*dx
    L.Assemble()

    gfu = GridFunction(fes)
    gfu.Set(uD, definedon=mesh.Boundaries("left"))
    gfu.vec.data += a.mat.Inverse(freedofs=fes.FreeDofs()) * L.vec
    return gfu

In [126]:

nu = 0.01 
beta = CF((100*exp(-(y-0.5)**2/0.1**2),.0))
# beta = CF((1,.0))
uD = 10*exp(-(y-0.5)**2/0.1**2)
u = solve_advdiff_2(order=1, h_max=0.025, uD=uD, nu=nu, beta=beta)

In [127]:
Draw(u)


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene

In [106]:
ngmesh = unit_square.GenerateMesh(maxh=0.05)
mesh = Mesh(ngmesh)
uD = exp(-(y-0.5)**2/0.1**2)
Draw(beta,mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene